<a href="https://colab.research.google.com/github/jawahirkhaleel/-jawahirkhaleel-/blob/main/xee_hvplot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade xee
!pip install -U geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.1/477.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.5.24
    Uninstalling earthengine-api-1.5.24:
      Successfully uninstalled earthengine-api-1.5.24
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.1 MB/s eta 0:00:00
  Attempting uninstall: geemap
    Found existing installation: geemap 0.35.3
    Uninstalling geemap-0.35.3:
      Successfully uninstalled geemap-0.35.3


In [2]:
import ee

In [3]:
ee.Authenticate()
ee.Initialize(
    project = 'ee-jawahirkhaleel05',
    opt_url = 'https://earthengine-highvolume.googleapis.com'
)


In [4]:
import geemap


In [5]:
map = geemap.Map()
map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [6]:
roi = map.draw_last_feature.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          7.69043,
                          42.90816
                        ],
                        [
                          8.876953,
                          44.871443
                        ],
                        [
                          10.50293,
                          46.377254
                        ],
                        [
                          13.095703,
                          46.286224
                        ],
                        [
                          15.908203,
                          43.100983
                        ],
                        [
                          18.632813,
                          41.112469
                        ],
                        [
                          19.467773,
                          38.582526
                        ],
                        [
                          17.841797,
                          34.161818
                        ],
                        [
                          12.963867,
                          36.279707
                        ],
                        [
                          9.931641,
                          38.23818
                        ],
                        [
                          6.987305,
                          38.23818
                        ],
                        [
                          7.075195,
                          40.880295
                        ],
                        [
                          7.69043,
                          42.90816
                        ]
                      ]
                    ]
                  },
                  "geodesic": {
                    "constantValue": false
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [8]:
viirs = (
    ee.ImageCollection("NASA/VIIRS/002/VNP13A1")
    .filterDate('2025','2026')
    .select('EVI')
)

In [9]:
viirs

In [10]:
import xarray as xr

In [11]:
ds = xr.open_dataset(
    viirs,
    engine = 'ee',
    crs = 'EPSG:4326',
    geometry = roi,
    scale = 0.1
)

In [12]:
ds = ds.sortby('time') * 1

In [13]:
ds

<xarray.Dataset> Size: 3MB
Dimensions:  (time: 46, lon: 125, lat: 122)
Coordinates:
  * time     (time) datetime64[ns] 368B 2025-01-01 2025-01-09 ... 2025-12-27
  * lon      (lon) float64 1kB 7.037 7.137 7.237 7.337 ... 19.24 19.34 19.44
  * lat      (lat) float64 976B 34.21 34.31 34.41 34.51 ... 46.11 46.21 46.31
Data variables:
    EVI      (time, lon, lat) float32 3MB 0.089 0.1055 0.1173 ... nan nan nan
Attributes:
    crs:      EPSG:4326

In [14]:
ds_mean = ds.mean(dim = 'time')

In [15]:
!pip install hvplot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 4.8 MB/s eta 0:00:00


In [16]:
import hvplot.xarray

In [17]:
ds_mean

<xarray.Dataset> Size: 63kB
Dimensions:  (lon: 125, lat: 122)
Coordinates:
  * lon      (lon) float64 1kB 7.037 7.137 7.237 7.337 ... 19.24 19.34 19.44
  * lat      (lat) float64 976B 34.21 34.31 34.41 34.51 ... 46.11 46.21 46.31
Data variables:
    EVI      (lon, lat) float32 61kB 0.09036 0.09784 0.112 ... 0.2934 0.3265
Attributes:
    crs:      EPSG:4326

In [18]:
ds_mean.EVI.hvplot(
    x = 'lon', y = 'lat', height = 500, cmap = 'RdYlGn', robust = True,
    title = 'Average EVI - 2025'
)

:Image   [lon,lat]   (EVI)

In [19]:
ds_mean.EVI.hvplot.quadmesh(
    x = 'lon', y = 'lat', cmap = 'RdYlGn', robust =True, height = 500
)


:QuadMesh   [lon,lat]   (EVI)

In [20]:
ds_mean.EVI.hvplot.contour(
    x = 'lon', y = 'lat', robust = True, cmap = 'RdYlGn', height = 500, levels = 10
)


:Contours   [lon,lat]   (EVI)

In [21]:
ds_mean.EVI.hvplot.contourf(
    x = 'lon', y = 'lat', cmap = 'RdYlGn', robust = True, height = 500, levels = 20
)

:Polygons   [lon,lat]   (EVI)

In [22]:
ds.EVI.hvplot(
    x = 'lon', y = 'lat', cmap = 'RdYlGn', robust =True, height = 500,
    groupby = 'time', widget_location = 'bottom', widget_type = 'scrubber'
)

Column
    [0] HoloViews(DynamicMap, height=500, sizing_mode='fixed', widget_location='bottom', widget_type='scrubber', width=700)
    [1] WidgetBox(align=('center', 'end'))
        [0] Player(end=45, width=550)